#Recall@50

CV 0.937, LB (Stepik) отдельных моделей 0.872-0.874, итог — ансамбль двух ранкеров.

Пайплайн:
1. считаю около 13 скоров по всему корпусу для каждого запроса: BM25 по полям, char n-grams, история из train, kNN по похожим запросам, query expansion, локация/гео, фильтры
2. пул около 500 кандидатов: top-300 по эвристике + top-50 каждого скора
3. два ранкера (HistGradientBoosting) по пулу: W на всех фичах и B только на тексте / локации / микрокатегории, обученный на «холодной» истории
4. итоговый top-50 — по среднему рангов W и B

Всё локально, без предобученных моделей. Библиотеки: numpy, pandas, scipy, scikit-learn, pyarrow, snowballstemmer.

## Setup

In [1]:
import gc, re, time, hashlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import snowballstemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.ensemble import HistGradientBoostingClassifier


class CFG:
    data_dir = 'data'
    seed = 42
    # валидация
    n_val = 3000
    n_rk = 5000            # запросы для обучения ранкера
    unseen_share = 0.64    # доля запросов val, чей текст убираем из истории
    # поиск кандидатов
    knn_k = 30
    desc_words = 300
    # пул кандидатов
    pool_k = 300           # топ по эвристике
    per_ch = 50            # топ по каждому скору
    batch = 50
    w_exp, geo_w, geo_km = 1.0, 0.2, 15.0
    # ранкер
    n_neg = 150
    gbm = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=50,
               l2_regularization=1.0, early_stopping=False, random_state=0)
    k = 50
    ens_w = 0.5            # вес модели W в ансамбле W + B


T0 = time.time()

## Text preprocessing

In [2]:
_stemmer = snowballstemmer.stemmer('russian')
_stem_cache = {}
TOKEN_RE = re.compile(r'[a-zа-я0-9]+')
STOP = set('и в во на с со по для от до из к ко о об а но или у за при без не же ли бы то это '
           'как что так все вс весь вы мы я он она они их его ее ваш наш'.split())


def stem(w):
    s = _stem_cache.get(w)
    if s is None:
        s = _stemmer.stemWord(w) if ('а' <= w[0] <= 'я') and len(w) > 3 else w  # латиницу и числа не стеммим
        _stem_cache[w] = s
    return s


def tokens(text):
    if not isinstance(text, str):
        return []
    return TOKEN_RE.findall(text.lower().replace('ё', 'е'))


def stem_text(text):
    return ' '.join(stem(t) for t in tokens(text) if t not in STOP)


def clean_text(text):
    return ' '.join(tokens(text))


# фильтры поиска: 'Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье'
# ['Тип услуги Маникюр, педикюр', 'Вид услуги Красота, здоровье']
KEYS_RE = re.compile(r'(Тип услуги автосервиса|Тип услуги|Вид услуги|Онлайн-запись|Кто оказывает услуги)')
FILTER_KEYS = ('Тип услуги автосервиса', 'Тип услуги', 'Вид услуги')


def parse_filters(s):
    if not isinstance(s, str) or not s:
        return []
    parts = KEYS_RE.split(s)
    out = []
    for i in range(1, len(parts) - 1, 2):
        k, v = parts[i], parts[i + 1].strip()
        if k in FILTER_KEYS and v:
            out.append(f'{k} {v}')
    return out


print(stem_text('Ремонт механизма диванов на дому'), '|', parse_filters('Тип услуги Вид услуги Ремонт и отделка'))

ремонт механизм диван дом | ['Вид услуги Ремонт и отделка']


## Data

In [3]:
GROUP_COLS = ['search_query', 'search_location_id', 'search_infm_params_text',
              'search_category', 'search_is_delivery_search']


def prep_queries(df):
    df = df.copy()
    df['search_infm_params_text'] = df['search_infm_params_text'].fillna('')
    df['q_stem'] = df['search_query'].map(stem_text)
    df['q_clean'] = df['search_query'].map(clean_text)
    df['p_stem'] = df['search_infm_params_text'].map(stem_text)
    df['filters'] = df['search_infm_params_text'].map(parse_filters)
    return df


# цена и координаты в parquet хранятся как DECIMAL, pyarrow отдаёт объекты decimal.Decimal поэтому приводим к float
DEC_COLS = ['item_price', 'item_latitude', 'item_longitude']

items = pd.read_parquet(f'{CFG.data_dir}/benchmark_items.parquet')
items[DEC_COLS] = items[DEC_COLS].astype(float)
for c in ['item_title_raw', 'item_infm_params_text', 'item_description_raw']:
    items[c] = items[c].fillna('')
items['t_stem'] = items['item_title_raw'].map(stem_text)
items['t_clean'] = items['item_title_raw'].map(clean_text)
items['p_stem'] = items['item_infm_params_text'].map(stem_text)
# берём первые 300 слов описания, дальше в основном SEO-хвосты
items['d_stem'] = items['item_description_raw'].map(lambda s: ' '.join(stem_text(s[:4000]).split()[:CFG.desc_words]))
items.drop(columns=['item_description_raw'], inplace=True)

# описания объявлений из train не нужны
train = pd.read_parquet(f'{CFG.data_dir}/train.parquet',
                        columns=GROUP_COLS + ['item_id', 'item_title_raw', 'item_microcat_id', 'item_location_id',
                                              'item_latitude', 'item_longitude'])
train[DEC_COLS[1:]] = train[DEC_COLS[1:]].astype(float)
train = prep_queries(train)
train['t_stem'] = train['item_title_raw'].map(stem_text)

test = prep_queries(pd.read_parquet(f'{CFG.data_dir}/benchmark_queries.parquet'))
print(items.shape, train.shape, test.shape, f'{time.time() - T0:.0f}s')

(189212, 17) (497673, 16) (2452, 10) 82s


## EDA

In [4]:
corpus_ids = set(items.item_id)
print('same location (search == item):', round((train.search_location_id == train.item_location_id).mean(), 3))

sub = train[(train.filters.map(len) > 0) & train.item_id.isin(corpus_ids)]
params = items.set_index('item_id').item_infm_params_text
print('item matches all search filters:', round(np.mean([all(f in params[i] for f in fl) for fl, i in zip(sub.filters, sub.item_id)]), 3))
del sub, params

print('train items present in corpus:', train.item_id[train.item_id.isin(corpus_ids)].nunique(), '/', train.item_id.nunique())
tq = set(train.search_query.str.lower().str.strip())
print('test queries seen in train:', round(test.search_query.str.lower().str.strip().isin(tq).mean(), 3))
print('query length, words: train', round(train.search_query.str.split().map(len).mean(), 2),
      '/ test', round(test.search_query.str.split().map(len).mean(), 2))
print('queries with filters: train', round(train.filters.map(len).gt(0).mean(), 3),
      '/ test', round(test.filters.map(len).gt(0).mean(), 3))

same location (search == item): 0.831


item matches all search filters: 0.971


train items present in corpus: 18142 / 344825


test queries seen in train: 0.37


query length, words: train 2.34 / test 3.2
queries with filters: train 0.644 / test 0.34


Что из этого следует:
1. локация очень важна, но не ключевую роль играет: поиск по области ведёт в города области. Беру `P(item_loc | search_loc)` по train и расстояние по координатам
2. фильтры выполняются в 97% случаев, использую как множитель, а не отсечение
3. объявлений из train в корпусе всего 18k из 345k, так что прямая история помогает только части запросов


## Validation

Запрос = группа по `GROUP_COLS`, таргет = все выбранные в группе `item_id`. Recall честно считается только для групп, где все выбранные объявления есть в корпусе.

Первый вариант (случайные группы) давал 82% запросов, чей текст уже был в истории, а в тесте таких 36%, и CV сильно завышался. Поэтому:
1. сэмплирую тексты запросов равномерно, а не по частоте
2. для 64% из них выкидываю из истории все строки с этим текстом
3. `val` 3000 запросов для оценки, `rk` 5000 для обучения ранкера, всё остальное (`fit`) история для статистик

Дополнительно считаю recall с весами, чтобы доля запросов с фильтрами была как в тесте (34/66).

In [5]:
def make_split(train, corpus_ids):
    tr = train.copy()
    tr['gid'] = tr.groupby(GROUP_COLS, sort=False, dropna=False).ngroup()
    tr['in_corpus'] = tr['item_id'].isin(corpus_ids)
    tr['ql'] = tr['search_query'].str.lower().str.strip()
    all_in = tr.groupby('gid')['in_corpus'].all()
    g = tr.drop_duplicates('gid').set_index('gid')[['ql']]
    n_groups = g['ql'].value_counts()
    g = g[all_in.reindex(g.index).values]
    one = g.reset_index().groupby('ql')['gid'].first()  # одна группа на текст

    texts = np.array(sorted(one.index))
    rng = np.random.RandomState(CFG.seed)
    rng.shuffle(texts)
    picked = {'val': texts[:CFG.n_val], 'rk': texts[CFG.n_val:CFG.n_val + CFG.n_rk]}

    hold, drop_texts, out = set(), set(), {}
    for name, tx in picked.items():
        gids = one.loc[tx].values
        hold |= set(gids)
        # невиданными делаем только редкие тексты (<= 20 групп), чтобы не терять много истории
        cand = [t for t in tx if n_groups[t] <= 20]
        k = int(round(CFG.unseen_share * len(tx)))
        drop_texts |= set(rng.choice(cand, size=min(k, len(cand)), replace=False))
        out[name] = gids

    mask = tr['gid'].isin(hold) | tr['ql'].isin(drop_texts)
    fit = tr[~mask].drop(columns=['ql']).reset_index(drop=True)

    def queries(gids):
        part = tr[tr['gid'].isin(set(gids))]
        q = part.drop_duplicates('gid').set_index('gid')[GROUP_COLS + ['q_stem', 'q_clean', 'p_stem', 'filters']]
        q['relevant'] = part.groupby('gid')['item_id'].agg(lambda s: sorted(set(s)))
        return q.reset_index()

    return fit, queries(out['val']), queries(out['rk'])


fit, val, rk = make_split(train, corpus_ids)
fq = set(fit.search_query.str.lower().str.strip())
for name, q in [('val', val), ('rk', rk), ('test', test)]:
    print(f'{name:5s} n={len(q):5d}  seen={q.search_query.str.lower().str.strip().isin(fq).mean():.3f}  '
          f'with_filters={q.filters.map(len).gt(0).mean():.3f}')

hf = val.filters.map(len).gt(0).values
VAL_W = np.where(hf, 0.34 / hf.mean(), 0.66 / (1 - hf.mean()))


def report(r, name):
    print(f'{name:28s} recall@50 {r.mean():.4f}  weighted {np.average(r, weights=VAL_W):.4f}')

val   n= 3000  seen=0.293  with_filters=0.541


rk    n= 5000  seen=0.294  with_filters=0.542


test  n= 2452  seen=0.314  with_filters=0.340


## Retrieval

Все скоры считаются по всему корпусу, батчами по 50 запросов (dense 50 x 189k).

| score | что считает |
|---|---|
| `bm25_title / params / desc / all` | BM25 по стемам; `all` = title x3 + params + desc |
| `char_title` | cosine tf-idf по char 3-5 граммам заголовка, ловит опечатки и формы слов |
| `hist_text` | BM25 по текстам запросов, по которым объявление выбирали в истории |
| `knn_item` | объявления, выбранные по 30 ближайшим запросам истории |
| `exp_title / exp_all` | query expansion: tf-idf заголовков, выбранных по ближайшим запросам (включая объявления не из корпуса), cosine с объявлением |
| `p_mc` | распределение микрокатегорий по ближайшим запросам |
| `p_loc` | `P(item_loc \| search_loc)` по истории |
| `dist` | км от точки локации поиска до объявления |
| `filt` | доля фильтров поиска, которые есть в параметрах объявления |

Индексы по корпусу (`ItemIndex`) не зависят от train, строятся один раз. Статистики по истории (`Signals`) строятся дважды: по `fit` для валидации и по всему train для сабмита.

In [6]:
class BM25:
    def __init__(self, k1=1.2, b=0.75, min_df=1):
        self.k1, self.b, self.min_df = k1, b, min_df

    def fit(self, docs):
        self.cv = CountVectorizer(analyzer=str.split, min_df=self.min_df, dtype=np.float32)
        tf = self.cv.fit_transform(docs).tocsr()
        n = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        idf = np.log1p((n - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(1)).ravel()
        avg = dl.mean() if dl.mean() > 0 else 1.0
        denom = self.k1 * (1 - self.b + self.b * dl / avg)
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))
        data = tf.data * (self.k1 + 1) / (tf.data + denom[rows]) * idf[tf.indices]
        self.D_T = sp.csr_matrix((data.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape).T.tocsr()
        return self

    def score(self, qdocs):
        q = self.cv.transform(qdocs).tocsr()
        q.data[:] = 1.0  # повтор слова в запросе не учитываем дважды
        return (q @ self.D_T).toarray()


class CharTfidf:
    def __init__(self, ngram_range=(3, 5), min_df=2):
        self.vec = TfidfVectorizer(analyzer='char_wb', ngram_range=ngram_range, min_df=min_df,
                                   sublinear_tf=True, dtype=np.float32)

    def fit(self, docs):
        self.D_T = self.vec.fit_transform(docs).T.tocsr()
        return self

    def score(self, qdocs):
        return (self.vec.transform(qdocs) @ self.D_T).toarray()


WIN_RE = re.compile(r'(?:Тип услуги|Вид услуги)')


def compact_params(p):
    # оставляем только окна после 'Вид услуги' / 'Тип услуги': памяти в ~10 раз меньше, совпадения фильтров те же
    return ' | '.join(p[m.start():m.start() + 200] for m in WIN_RE.finditer(p))


class ItemIndex:
    # всё, что зависит только от корпуса
    def __init__(self, items):
        it = items.reset_index(drop=True)
        self.n = len(it)
        self.id2idx = pd.Series(np.arange(self.n), index=it['item_id'].values)

        self.bm_title = BM25(k1=1.2, b=0.5).fit(it['t_stem'])
        self.bm_params = BM25(k1=1.2, b=0.75).fit(it['p_stem'])
        self.bm_desc = BM25(k1=1.2, b=0.75, min_df=2).fit(it['d_stem'])
        t3 = it['t_stem'] + ' ' + it['t_stem'] + ' ' + it['t_stem'] + ' '  # заголовок x3
        self.bm_all = BM25(k1=1.5, b=0.75, min_df=2).fit(t3 + it['p_stem'] + ' ' + it['d_stem'])
        del t3
        self.ch_title = CharTfidf().fit(it['t_clean'])
        # tf-idf пространства для расширения запроса
        self.tv_title = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, min_df=2, dtype=np.float32)
        self.TD_title = self.tv_title.fit_transform(it['t_stem']).T.tocsr()
        self.tv_all = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, min_df=3, max_df=0.3, dtype=np.float32)
        self.TD_all = self.tv_all.fit_transform(it['t_stem'] + ' ' + it['p_stem'] + ' ' + it['d_stem']).T.tocsr()

        self.item_params = np.array([compact_params(x) for x in it['item_infm_params_text'].values], dtype=object)
        self._filt_cache = {}

        keep = ['item_id', 'item_title_raw', 'item_microcat_id', 'item_location_id', 'item_latitude',
                'item_longitude', 'item_price', 'item_rating', 'item_rating_reviews_count',
                'item_is_phone_hidden', 'item_is_message_forbidden']
        self.items = it[keep].copy()
        self.item_loc = self.items['item_location_id'].values
        self.loc_codes, self.loc_uni = pd.factorize(self.items['item_location_id'])

In [7]:
class Signals:
    # индекс корпуса + статистики по истории (fit для CV, весь train для сабмита)
    def __init__(self, index, hist, knn_k=CFG.knn_k):
        self.__dict__.update(index.__dict__)  # общие матрицы корпуса, без копирования
        self.knn_k = knn_k
        it = self.items

        # запросы, по которым выбирали каждое объявление корпуса
        h = hist[hist['item_id'].isin(self.id2idx.index)]
        htxt = h.groupby('item_id')['q_stem'].agg(' '.join)
        hdocs = pd.Series('', index=it['item_id'].values)
        hdocs.loc[htxt.index] = htxt.values
        self.bm_hist = BM25(k1=1.2, b=0.3).fit(hdocs.values)
        pop = h.groupby('item_id').size()
        self.item_pop = np.zeros(self.n, np.float32)
        self.item_pop[self.id2idx[pop.index].values] = pop.values

        # kNN по уникальным текстам запросов истории (tf-idf по словам и символам)
        hq = hist[['q_stem', 'q_clean', 'item_microcat_id', 'item_id', 'search_location_id']]
        texts = hq.drop_duplicates('q_stem').reset_index(drop=True)[['q_stem', 'q_clean']]
        t2i = pd.Series(np.arange(len(texts)), index=texts['q_stem'].values)
        self.knn_word = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, dtype=np.float32).fit(texts['q_stem'])
        self.knn_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=2,
                                        sublinear_tf=True, dtype=np.float32).fit(texts['q_clean'])
        self.K_T = self._qvec(texts['q_stem'], texts['q_clean']).T.tocsr()

        # текст распределение микрокатегорий
        mc_codes, self.mc_uni = pd.factorize(hq['item_microcat_id'])
        ti = t2i[hq['q_stem'].values].values
        self.T_mc = normalize(sp.csr_matrix((np.ones(len(hq), np.float32), (ti, mc_codes)),
                                            shape=(len(texts), len(self.mc_uni))), norm='l1')
        mc_map = pd.Series(np.arange(len(self.mc_uni)), index=self.mc_uni)
        self.item_mc = mc_map.reindex(it['item_microcat_id'].values).fillna(-1).astype(int).values

        # текст выбранные объявления корпуса
        inc = hq['item_id'].isin(self.id2idx.index).values
        self.T_item = sp.csr_matrix(
            (np.ones(inc.sum(), np.float32), (ti[inc], self.id2idx[hq['item_id'].values[inc]].values)),
            shape=(len(texts), self.n))
        self.T_item.sum_duplicates()
        self.T_item.data = np.minimum(self.T_item.data, 3)

        # расширение запроса: заголовки ВСЕХ объявлений, выбранных по тексту, включая объявления не из корпуса
        # ('сауна' в выбранных заголовках часто встречается 'баня')
        exp_docs = hist.groupby('q_stem')['t_stem'].agg(' '.join).reindex(texts['q_stem']).fillna('')
        self.E_title = normalize(self.tv_title.transform(exp_docs.values))
        self.E_all = normalize(self.tv_all.transform(exp_docs.values))
        del exp_docs, texts, hq

        # P(локация объявления | локация поиска)
        c = hist.groupby(['search_location_id', 'item_location_id']).size()
        self.loc_p = (c / c.groupby(level=0).transform('sum')).to_dict()

        # координаты локации = медиана координат объявлений в ней; для локации-региона - медиана выбранных объявлений
        lat = pd.concat([it[['item_location_id', 'item_latitude', 'item_longitude']],
                         hist[['item_location_id', 'item_latitude', 'item_longitude']]])
        lc = lat.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()
        sc = hist.groupby('search_location_id')[['item_latitude', 'item_longitude']].median()
        self.search_coord = sc.combine_first(lc).to_dict('index')
        self.search_coord.update(lc.to_dict('index'))
        il = it['item_latitude'].fillna(it['item_location_id'].map(lc['item_latitude'])).fillna(0)
        io = it['item_longitude'].fillna(it['item_location_id'].map(lc['item_longitude'])).fillna(0)
        self.item_lat = np.radians(il.values.astype(np.float64))
        self.item_lon = np.radians(io.values.astype(np.float64))

    def _qvec(self, q_stem, q_clean):
        return normalize(sp.hstack([self.knn_word.transform(q_stem), self.knn_char.transform(q_clean) * 0.7]).tocsr())

    def knn(self, q):
        S = (self._qvec(q['q_stem'], q['q_clean']) @ self.K_T).toarray()
        idx = np.argpartition(-S, self.knn_k, axis=1)[:, :self.knn_k]
        w = np.take_along_axis(S, idx, 1)
        return idx, np.where(w > 0.2, w, 0) ** 2

    @staticmethod
    def _expand(W, E, TD, top_terms=40):
        X = normalize((W @ E).tocsr())
        for i in range(X.shape[0]):  # оставляем top-40 слов расширения на запрос
            a, b = X.indptr[i], X.indptr[i + 1]
            if b - a > top_terms:
                d = X.data[a:b]
                thr = np.partition(d, -top_terms)[-top_terms]
                X.data[a:b] = np.where(d >= thr, d, 0)
        X.eliminate_zeros()
        return (normalize(X) @ TD).toarray().astype(np.float32)

    def loc_row(self, loc):
        p = np.zeros(len(self.loc_uni), np.float32)
        for j, L in enumerate(self.loc_uni):
            v = self.loc_p.get((loc, L))
            if v is not None:
                p[j] = v
        if p.sum() == 0:  # незнакомая локация поиска
            p[self.loc_uni == loc] = 0.8
        return p[self.loc_codes]

    def dist_row(self, loc):
        c = self.search_coord.get(loc)
        if c is None or pd.isna(c['item_latitude']):
            return np.full(self.n, 3000.0, np.float32)
        la, lo = np.radians(c['item_latitude']), np.radians(c['item_longitude'])
        h = (np.sin((self.item_lat - la) / 2) ** 2
             + np.cos(la) * np.cos(self.item_lat) * np.sin((self.item_lon - lo) / 2) ** 2)
        return (2 * 6371 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))).astype(np.float32)  # расстояние по формуле гаверсинусов, км

    def filt_mask(self, f):
        m = self._filt_cache.get(f)
        if m is None:
            m = np.fromiter((f in p for p in self.item_params), bool, self.n)
            self._filt_cache[f] = m
        return m

    def batch(self, q):
        # все скоры для батча запросов:(len(q), n_items)
        qs, qc = q['q_stem'].values, q['q_clean'].values
        out = {
            'bm25_title': self.bm_title.score(qs),
            'bm25_params': self.bm_params.score(qs),
            'bm25_desc': self.bm_desc.score(qs),
            'bm25_all': self.bm_all.score(qs),
            'char_title': self.ch_title.score(qc),
            'hist_text': self.bm_hist.score(qs),
        }
        idx, w = self.knn(q)
        B = len(q)
        W = sp.csr_matrix((w.ravel(), (np.repeat(np.arange(B), idx.shape[1]), idx.ravel())),
                          shape=(B, self.K_T.shape[1]))
        out['knn_item'] = (W @ self.T_item).toarray()
        out['exp_title'] = self._expand(W, self.E_title, self.TD_title)
        out['exp_all'] = self._expand(W, self.E_all, self.TD_all)
        mc = (W @ self.T_mc).toarray()
        s = mc.sum(1, keepdims=True)
        mc = np.where(s > 0, mc / np.maximum(s, 1e-9), 0).astype(np.float32)
        out['p_mc'] = mc[:, np.maximum(self.item_mc, 0)]
        out['p_mc'][:, self.item_mc < 0] = 0

        locs = q['search_location_id'].values
        lcache = {L: self.loc_row(L) for L in set(locs)}
        dcache = {L: self.dist_row(L) for L in set(locs)}
        out['p_loc'] = np.stack([lcache[L] for L in locs])
        out['dist'] = np.stack([dcache[L] for L in locs])

        F = np.ones((B, self.n), np.float32)
        for i, fl in enumerate(q['filters'].values):
            if fl:
                F[i] = np.mean([self.filt_mask(f) for f in fl], axis=0)
        out['filt'] = F
        return out

In [8]:
t = time.time()
IX = ItemIndex(items)
titles = IX.items.set_index('item_id').item_title_raw
del items; gc.collect()
S = Signals(IX, fit)
del fit; gc.collect()
print(f'index built in {time.time() - t:.0f}s')

index built in 122s


## Candidates

Эвристика подобрана на валидации: сумма нормированных текстовых скоров, умноженная на локацию, микрокатегорию и фильтры

`combo = text * (0.05 + p_loc + 0.2 * exp(-dist / 15)) ** 0.5 * (0.05 + p_mc) ** 0.3 * (0.1 + filt)`

Гео-слагаемое нужно для пар соседних локаций, которых не было в истории: без него `p_loc = 0` убирал кандидата.

In [9]:
TEXT_CH = ['bm25_title', 'bm25_params', 'bm25_desc', 'bm25_all', 'hist_text', 'knn_item']
ALL_CH = TEXT_CH + ['char_title', 'exp_title', 'exp_all', 'p_mc', 'p_loc', 'filt', 'dist']
NO_TOPK = ('p_loc', 'filt', 'dist')  # сами по себе кандидатов не дают
RANK_CH = ['combo', 'bm25_all', 'bm25_title', 'char_title', 'bm25_desc', 'hist_text', 'knn_item', 'p_mc',
           'exp_title', 'exp_all', 'dist']


def rownorm(x):
    return (x / np.maximum(x.max(1, keepdims=True), 1e-9)).astype(np.float32)


def combo_score(ch):
    text = (rownorm(ch['bm25_all']) + rownorm(ch['bm25_title']) + ch['char_title']
            + 0.5 * rownorm(ch['hist_text']) + 0.5 * rownorm(ch['knn_item'])
            + CFG.w_exp * (ch['exp_title'] + ch['exp_all']))
    ploc = ch['p_loc'] + CFG.geo_w * np.exp(-ch['dist'] / CFG.geo_km)
    return text * (0.05 + ploc) ** 0.5 * (0.05 + ch['p_mc']) ** 0.3 * (0.1 + ch['filt'])


def topk_idx(a, k):
    return np.argpartition(-a, k, axis=1)[:, :k].copy()  # copy: иначе срез держит в памяти весь массив


def build_pool(S, qdf):
    it = S.items
    rating = it['item_rating'].fillna(0).values.astype(np.float32)
    nrev = np.log1p(it['item_rating_reviews_count'].fillna(0).values).astype(np.float32)
    price = np.log1p(it['item_price'].fillna(0).clip(lower=0).values).astype(np.float32)
    phone_h = it['item_is_phone_hidden'].astype(float).values.astype(np.float32)
    msg_f = it['item_is_message_forbidden'].astype(float).values.astype(np.float32)

    parts = []
    for s in range(0, len(qdf), CFG.batch):
        q = qdf.iloc[s:s + CFG.batch]
        ch = S.batch(q)
        ch['combo'] = combo_score(ch)
        cand = [topk_idx(ch['combo'], CFG.pool_k)] + [topk_idx(ch[c], CFG.per_ch) for c in ALL_CH if c not in NO_TOPK]
        cand = np.concatenate(cand, axis=1)
        rows = [np.unique(cand[i]) for i in range(len(q))]
        qi = np.concatenate([np.full(len(r), i) for i, r in enumerate(rows)])
        ii = np.concatenate(rows)

        f = {'qi': qi + s, 'item': ii}
        for c in ALL_CH + ['combo']:
            f[c] = ch[c][qi, ii]
        for c in TEXT_CH + ['combo']:
            f[c + '_n'] = f[c] / np.maximum(ch[c].max(1)[qi], 1e-9)
        f['combo_qmax'] = ch['combo'].max(1)[qi]
        f['item_pop'] = S.item_pop[ii]
        f['rating'] = rating[ii]
        f['nrev'] = nrev[ii]
        f['price'] = price[ii]
        f['phone_hidden'] = phone_h[ii]
        f['msg_forbidden'] = msg_f[ii]
        f['same_loc'] = (S.item_loc[ii] == q['search_location_id'].values[qi]).astype(np.float32)
        f['q_len'] = q['q_stem'].str.split().map(len).values[qi]
        f['q_nfilt'] = q['filters'].map(len).values[qi]
        f['q_cat0'] = (q['search_category'].values[qi] == 0).astype(np.float32)
        f['pmc_max'] = ch['p_mc'].max(1)[qi]
        df = pd.DataFrame(f)
        for c in df.columns:
            if c not in ('qi', 'item'):
                df[c] = df[c].astype(np.float32)
        parts.append(df)
        del ch

    pool = pd.concat(parts, ignore_index=True)
    for c in RANK_CH:
        pool[c + '_rk'] = pool.groupby('qi')[c].rank(ascending=(c == 'dist'), method='min').astype(np.float32)
    return pool


def add_labels(pool, qdf, id2idx):
    rel = {(i, int(id2idx[x])) for i, r in enumerate(qdf['relevant'].values) for x in r}
    pool['y'] = np.array([(a, b) in rel for a, b in zip(pool['qi'].values, pool['item'].values)], dtype=np.int8)
    return pool


def recall(top, qdf, id2idx):
    r = []
    for i, rel in enumerate(qdf['relevant'].values):
        got = set(top.get(i, [])[:CFG.k])
        r.append(np.mean([int(id2idx[x]) in got for x in rel]))
    return np.array(r)

Recall отдельных скоров (top-50 по одному скору, первые 1000 запросов val):

In [10]:
sub = val.iloc[:1000]
rel_idx = [S.id2idx[r].values for r in sub.relevant]
res = {}
for s in range(0, len(sub), CFG.batch):
    ch = S.batch(sub.iloc[s:s + CFG.batch])
    ch['combo'] = combo_score(ch)
    for c in [c for c in ALL_CH if c not in NO_TOPK] + ['combo']:
        res.setdefault(c, []).extend(topk_idx(ch[c], CFG.k))
    del ch
pd.Series({c: np.mean([len(set(t) & set(r)) / len(r) for t, r in zip(v, rel_idx)]) for c, v in res.items()},
          name='recall@50').round(4).to_frame()

,recall@50
bm25_title,0.2907
bm25_params,0.1312
bm25_desc,0.3291
bm25_all,0.3768
hist_text,0.2098
knn_item,0.1891
char_title,0.2730
exp_title,0.2335
exp_all,0.2017
p_mc,0.0408


In [11]:
t = time.time()
pool_val = add_labels(build_pool(S, val), val, S.id2idx)
pool_rk = add_labels(build_pool(S, rk), rk, S.id2idx)
id2idx = S.id2idx
print(f'pools built in {time.time() - t:.0f}s, {len(pool_val) / len(val):.0f} candidates per query')

n_rel = val.relevant.map(len).values
report(pool_val.groupby('qi').y.sum().reindex(range(len(val)), fill_value=0).values / n_rel, 'pool coverage')
combo_top = pool_val.sort_values(['qi', 'combo'], ascending=[True, False]).groupby('qi')['item'].agg(lambda s: list(s.iloc[:CFG.k]))
report(recall(combo_top, val, id2idx), 'heuristic top-50')

pools built in 381s, 494 candidates per query
pool coverage                recall@50 0.9750  weighted 0.9716


heuristic top-50             recall@50 0.8814  weighted 0.8750


## Ranker

Бинарная классификация «объявление выбрали / нет» на пуле `rk`: все позитивы + 150 случайных негативов на запрос. 44 фичи: сами скоры, их нормировка на максимум по запросу, ранги внутри пула, популярность в истории, рейтинг / отзывы / цена / флаги, совпадение локации, длина запроса, число фильтров.

In [12]:
FEATS = [c for c in pool_rk.columns if c not in ('qi', 'item', 'y')]
QUAL = ['rating', 'nrev', 'price', 'phone_hidden', 'msg_forbidden']
HIST = ['item_pop', 'hist_text', 'hist_text_n', 'hist_text_rk', 'knn_item', 'knn_item_n', 'knn_item_rk']
FEATS_B = [c for c in FEATS if c not in QUAL + HIST]  # признаки модели B: только текст, локация, микрокатегория


def sample_train(pool, seed):
    rng = np.random.RandomState(seed)
    r = pd.Series(rng.rand(len(pool)), index=pool.index)
    neg = pool['y'] == 0
    keep = (~neg) | (r[neg].groupby(pool.loc[neg, 'qi']).rank(method='first') <= CFG.n_neg).reindex(pool.index, fill_value=False)
    return pool[keep]


def train_ranker(pools, feats):
    tr = pd.concat([sample_train(p, seed=k).assign(qi=lambda d, k=k: d['qi'] + k * 10_000_000)
                    for k, p in enumerate(pools)], ignore_index=True)
    return HistGradientBoostingClassifier(**CFG.gbm).fit(tr[feats].values, tr['y'].values)


def top_by(p, col, ascending):
    p = p.sort_values(['qi', col, 'item'], ascending=[True, ascending, True], kind='mergesort')
    return p.groupby('qi').head(CFG.k).groupby('qi')['item'].agg(list)


def predict_top(pool, model, feats):
    p = pool[['qi', 'item']].copy()
    p['s'] = model.predict_proba(pool[feats].values)[:, 1]
    return top_by(p, 's', False)


model = train_ranker([pool_rk], FEATS)
r_val = recall(predict_top(pool_val, model, FEATS), val, id2idx)
report(r_val, 'ranker top-50 (CV)')
del S; gc.collect()

ranker top-50 (CV)           recall@50 0.9371  weighted 0.9310


63

## Error analysis

In [13]:
warm = pool_val[pool_val.y == 1].groupby('qi').item_pop.max().reindex(range(len(val))).fillna(0).values > 0
in_pool = pool_val.groupby('qi').y.sum().reindex(range(len(val)), fill_value=0).values / n_rel
an = pd.DataFrame({'recall': r_val, 'in_pool': in_pool, 'item_in_history': warm, 'has_filters': hf})
print(an.groupby('item_in_history')[['recall', 'in_pool']].agg(['mean', 'count']).round(3), '\n')
print(an.groupby('has_filters')[['recall', 'in_pool']].mean().round(3), '\n')

# permutation importance по группам фич
rng = np.random.RandomState(0)
groups = {
    'history': ['knn_item', 'knn_item_n', 'knn_item_rk', 'hist_text', 'hist_text_n', 'hist_text_rk', 'item_pop'],
    'query expansion': ['exp_title', 'exp_all', 'exp_title_rk', 'exp_all_rk'],
    'distance': ['dist', 'dist_rk'],
    'location stats': ['p_loc', 'same_loc'],
    'text': [c for c in FEATS if c.startswith(('bm25', 'char'))],
}
for g, cols in groups.items():
    pv = pool_val.copy()
    for c in cols:
        pv[c] = rng.permutation(pv[c].values)
    r = recall(predict_top(pv, model, FEATS), val, id2idx)
    print(f'shuffle {g:16s} recall {r.mean():.4f}  ({r.mean() - r_val.mean():+.4f})')
    del pv

                recall       in_pool      
                  mean count    mean count
item_in_history                           
False            0.903  1636   0.954  1636
True             0.978  1364   1.000  1364 

             recall  in_pool
has_filters                 
False         0.921    0.966
True          0.951    0.983 



shuffle history          recall 0.9264  (-0.0107)


shuffle query expansion  recall 0.9344  (-0.0027)


shuffle distance         recall 0.9009  (-0.0362)


shuffle location stats   recall 0.9249  (-0.0122)


shuffle text             recall 0.9016  (-0.0354)


In [14]:
miss = [(q.search_query, q.search_infm_params_text[:40], titles[q.relevant[0]])
        for i, q in enumerate(val.itertuples()) if r_val[i] == 0][:15]
pd.DataFrame(miss, columns=['query', 'filters', 'missed item title'])

,query,filters,missed item title
0,аренда ноутбука,"Вид услуги Праздники, мероприятия","Аренда, Прокат Компьютера"
1,сантехник колпино,,"Мастер на час, муж на час"
2,"террасы, веранды под ключ",,Строительство пристроек террас веранд бань гар...
3,компьютерный мастер,Вид услуги Компьютерная помощь,Установка Windows Ремонт компьютеров выезд
4,треугольный дом,"Вид услуги Праздники, мероприятия",Отдых на природе в доме a-frame с чаном
5,сауна с бассейном,,Русская баня на дровах
6,сдать iphone,Тип услуги Мобильные устройства Вид услу,Выкуп ваших смартфонов
7,ремонт мотоамортизаторов,,"Ремонт мотоциклов, скутеров, квадроциклов, эле..."
8,сантехник,Вид услуги Ремонт и отделка,Сантехник услуги отопление водоснабжения ремонт
9,сауна,"Онлайн-запись Тип услуги СПА-услуги, мас",Баня Снять Дом с Баней Баня с бассейном


Промахи:
1. объявления без истории в train: recall 0.90 против 0.98. Основной остаток ошибок
2. разрыв в словах: «сауна» vs «Русская баня на дровах», «вынести диван» vs «Вывоз старой мебели». Частично закрыто query expansion, дальше помогли бы эмбеддинги
3. общие запросы в больших городах («сантехник», «компьютерный мастер»): сотни одинаково подходящих объявлений, выбор пользователя почти случайный
4. соседние населённые пункты без статистики по паре локаций. Закрыл через `dist`, это самая важная группа фич

Что пробовал(CV):
1. hard negatives (top по эвристике) вместо случайных: 0.9360 vs 0.9362
2. все негативы / 300 на запрос: 0.9352
3. модель больше (600 итераций, lr 0.03, 127 листьев): 0.9316
4. вес query expansion 2.0 в эвристике вместо 1.0: 0.8816 vs 0.8866 (на 1500 запросах)

## CV vs LB и вторая модель

Первый сабмит этой модели дал на платформе 0.872 при CV 0.937. Чтобы понять разрыв, отправил несколько диагностических вариантов:

| вариант | CV | LB |
|---|---|---|
| эвристика `combo` без модели | 0.881 | 0.826 |
| ранкер (модель W, все фичи) | 0.937 | 0.872 |
| ранкер, обученный на «холодной» истории, все фичи | - | 0.706 |
| ранкер на «холодной» истории, без рейтинга / цены / истории (модель B) | - | 0.874 |

«Холодная» история: из истории убраны вообще все строки с объявлениями из `val` и `rk`, как будто это новые объявления.

Выводы:
1. разрыв CV/LB ~6 п.п. одинаковый у эвристики и у ранкера, значит модель не переобучена, тест просто сложнее валидации. Скорее всего в тесте у запросов чаще несколько правильных объявлений, а моя валидация из-за отбора (все выбранные объявления должны быть в корпусе) почти целиком из запросов с одним ответом
2. «холодный» ранкер со всеми фичами учится занижать объявления с историей и проваливается до 0.706, значит в тесте много объявлений, которые уже встречались в train, и история нужна
3. модель B, которая вообще не смотрит на историю и качество объявления, даёт тот же LB, что и W, но выдача у них заметно разная (пересечение top-50 ~86%)

Поэтому итог — ансамбль W и B: среднее геометрическое их рангов внутри пула. На обеих валидациях (обычной и «холодной») ансамбль почти не уступает лучшей из двух моделей, то есть он устойчив к тому, какой из сценариев ближе к тесту.

In [15]:
# финальная модель W: переобучаем на rk + val
final_W = train_ranker([pool_rk, pool_val], FEATS)
# для оценки ансамбля на обычной val оставляем только нужные колонки
pool_val_w = pool_val[['qi', 'item', 'y'] + FEATS_B].copy()
pool_val_w['sW'] = model.predict_proba(pool_val[FEATS].values)[:, 1]
del pool_rk, pool_val; gc.collect()

# объявления из val и rk полностью убраны из истории
fit, _, _ = make_split(train, corpus_ids)
hold_items = {x for r in val.relevant for x in r} | {x for r in rk.relevant for x in r}
fit_cold = fit[~fit.item_id.isin(hold_items)].reset_index(drop=True)
print('history rows:', len(fit), '-> cold:', len(fit_cold))
del fit

t = time.time()
S = Signals(IX, fit_cold)
del fit_cold; gc.collect()
pool_val_c = add_labels(build_pool(S, val), val, id2idx)
pool_rk_c = add_labels(build_pool(S, rk), rk, id2idx)
del S; gc.collect()
print(f'cold pools built in {time.time() - t:.0f}s')

model_B = train_ranker([pool_rk_c], FEATS_B)


def ens_top(pool, sW, sB):
    # ансамбль: взвешенное среднее логарифмов рангов двух моделей внутри пула
    p = pool[['qi', 'item']].copy()
    p['rW'] = pd.Series(sW, index=p.index).groupby(p['qi']).rank(ascending=False)
    p['rB'] = pd.Series(sB, index=p.index).groupby(p['qi']).rank(ascending=False)
    p['e'] = CFG.ens_w * np.log(p['rW']) + (1 - CFG.ens_w) * np.log(p['rB'])
    return top_by(p, 'e', True)


sB_w = model_B.predict_proba(pool_val_w[FEATS_B].values)[:, 1]
sW_c = model.predict_proba(pool_val_c[FEATS].values)[:, 1]
sB_c = model_B.predict_proba(pool_val_c[FEATS_B].values)[:, 1]
rows = []
for name, pool, sW, sB in [('val', pool_val_w, pool_val_w['sW'].values, sB_w), ('cold val', pool_val_c, sW_c, sB_c)]:
    p = pool[['qi', 'item']].assign(sW=sW, sB=sB)
    rows.append({'validation': name,
                 'W': recall(top_by(p, 'sW', False), val, id2idx).mean(),
                 'B': recall(top_by(p, 'sB', False), val, id2idx).mean(),
                 'ensemble': recall(ens_top(pool, sW, sB), val, id2idx).mean()})
pd.DataFrame(rows).set_index('validation').round(4)

history rows: 468185 -> cold: 461823


cold pools built in 395s


,W,B,ensemble
validation,,,
val,0.9371,0.9299,0.9367
cold val,0.9199,0.9246,0.9232


## Submission

Обе модели переобучаю на `rk + val` (8000 запросов, параметры те же), статистики истории пересчитываю по всему train, итог — ансамбль.

In [16]:
final_B = train_ranker([pool_rk_c, pool_val_c], FEATS_B)
del pool_rk_c, pool_val_c, pool_val_w; gc.collect()

t = time.time()
S = Signals(IX, train)
pool_test = build_pool(S, test)
top_test = ens_top(pool_test,
                   final_W.predict_proba(pool_test[FEATS].values)[:, 1],
                   final_B.predict_proba(pool_test[FEATS_B].values)[:, 1])
ids = S.items.item_id.values
sub = pd.DataFrame({'query_id': test.query_id.values,
                    'answer': [' '.join(ids[top_test[i]]) for i in range(len(test))]})
sub.to_csv('answer.csv', index=False)
print(f'done in {time.time() - t:.0f}s, total {time.time() - T0:.0f}s')
sub.assign(answer=sub.answer.str[:120] + ' ...').head()

done in 163s, total 1544s


,query_id,answer
0,70DfDUpwjxB4lzFd,255fbeaf526a1cc1 deb58d7ad8c27d06 b92ee8f432ce...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 1344796085e92421 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 3f89b8062dc85f1c dab52187b450...
3,660ac9QVtXkRxZC3,4c15e7abba063ac5 af91ec4a29b66636 3cce43c81145...
4,YgHcM9MVbxKnxD1e,1a62634394544781 9c41fe789f98d702 60e3b99111bd...


In [17]:
# проверки формата
a = pd.read_csv('answer.csv', dtype=str, keep_default_na=False)
lst = a.answer.str.split(' ')
assert list(a.columns) == ['query_id', 'answer']
assert a.query_id.is_unique and set(a.query_id) == set(test.query_id)
assert a.query_id.str.len().eq(16).all()
assert lst.map(len).eq(50).all()
assert lst.map(lambda x: len(x) == len(set(x))).all()
assert all(x in corpus_ids for l in lst for x in l)
assert all(re.fullmatch(r'[0-9a-f]{16}', x) for l in lst for x in l)
print('ok, md5', hashlib.md5(open('answer.csv', 'rb').read()).hexdigest())

ok, md5 5e3d1763c510bbbc972e5974b2992aa6
